In [1]:
import pandas as pd

df = pd.read_csv("../detected.csv")

df.head(5)

,timestamp,method,path,status,ip,port,uuid,sess_uuid,detection_name,detection_type,user_agent,referer,attack_name,attack_type
0,2025-05-22T07:53:51.660458,GET,/,200,172.18.0.1,38158,5da851cd-09cc-4bb7-a200-f07dad1a8897,e906dd52-8d43-4894-98e0-3b6b93073434,index,1,curl/7.81.0,NaN,Unknown,0
1,2025-05-22T07:54:41.725206,GET,/,200,172.18.0.1,51798,5da851cd-09cc-4bb7-a200-f07dad1a8897,4071c0d0-c143-4171-9d72-57001462d87e,index,1,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,Unknown,0
2,2025-05-22T07:54:41.929057,GET,/stylesheets/jquery/jquery-ui-1.11.0.css?15286...,200,172.18.0.1,51806,5da851cd-09cc-4bb7-a200-f07dad1a8897,c959aef7-3a6f-4f11-8898-6c8402172368,index,1,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,http://localhost/,Unknown,0
3,2025-05-22T07:54:42.114829,GET,/stylesheets/application.css?1528612569,200,172.18.0.1,51812,5da851cd-09cc-4bb7-a200-f07dad1a8897,c959aef7-3a6f-4f11-8898-6c8402172368,index,1,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,http://localhost/,Unknown,0
4,2025-05-22T07:54:42.131187,GET,/stylesheets/responsive.css?1528612569,200,172.18.0.1,51826,5da851cd-09cc-4bb7-a200-f07dad1a8897,c959aef7-3a6f-4f11-8898-6c8402172368,index,1,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,http://localhost/,Unknown,0


In [2]:
method = df["method"]

methods_matrix = []
tmp = []

for i, x in enumerate(method):
  if i % 10 == 0 and i != 0:
    methods_matrix.append(tmp)
    tmp = []
  tmp.append(x)

# Add the last group if not empty
if tmp:
  methods_matrix.append(tmp)

methods_matrix

[['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'POST', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET'],
 ['GET', 'GET', 'GET', 'GET', 'GET', 'GET', 'GET',

In [3]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, fpmax, fpgrowth

te = TransactionEncoder()
te_ary = te.fit(methods_matrix).transform(methods_matrix)
df = pd.DataFrame(te_ary, columns=te.columns_)

df

,GET,HEAD,OPTIONS,POST,PROPFIND,PUT,SEARCH,TRACE
0,True,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False
2,True,False,False,True,False,False,False,False
3,True,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
4926,True,False,False,False,False,False,False,False
4927,True,False,False,False,False,False,False,False
4928,True,False,False,False,False,False,False,False
4929,True,False,False,False,False,False,False,False


In [8]:
# frequent_itemsets = fpgrowth(df, min_support=0.3, use_colnames=True)
frequent_itemsets = fpgrowth(df, min_support=0.0001, use_colnames=True)
### alternatively:
#frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
#frequent_itemsets = fpmax(df, min_support=0.6, use_colnames=True)

frequent_itemsets.head(20)

,support,itemsets
0,0.991077,(GET)
1,0.468870,(POST)
2,0.000608,(HEAD)
3,0.001014,(OPTIONS)
4,0.000608,(PUT)
5,0.000608,(TRACE)
6,0.000608,(PROPFIND)
7,0.000203,(SEARCH)
8,0.459947,"(POST, GET)"
9,0.000608,"(HEAD, GET)"


In [9]:
import psycopg2

conn = psycopg2.connect(database="web_honeypot_generated", user = "postgres", password = "admin", host = "127.0.0.1", port = "5432")

print("Opened database successfully")

Opened database successfully


In [10]:
# create table
cur = conn.cursor()
cur.execute('''CREATE TABLE assoc_rules_methods (
            ID INT PRIMARY KEY     NOT NULL,
            SUPPORT           REAL    NOT NULL,
            METHOD            VARCHAR(255)     NOT NULL);''')

print("Table created successfully")

conn.commit()

Table created successfully


In [11]:
# insert data

cur = conn.cursor()

insert_query = """
    INSERT INTO assoc_rules_methods (ID, SUPPORT, METHOD)
    VALUES (%s, %s, %s)
"""

for idx, row in frequent_itemsets.iterrows():
    cur.execute(
        insert_query,
        (int(idx), float(row["support"]), str(list(row["itemsets"])))
    )

conn.commit()
print("Records created successfully")
conn.close()

Records created successfully
